In [1]:
#!/usr/bin/env python3
"""
Network Probe Tool (Ping-Based)
Educational Use Only
"""

import subprocess
import socket
import platform
import datetime
import sys

# ─────────────────────────────────────────────
# HEADER DISPLAY
# ─────────────────────────────────────────────
def display_header():
    print("=" * 65)
    print("        NETWORK PROBE TOOL - USING PING")
    print("        (Educational Purpose Only)")
    print("=" * 65)


# ─────────────────────────────────────────────
# DOMAIN TO IP
# ─────────────────────────────────────────────
def fetch_ip(target):
    try:
        ip_addr = socket.gethostbyname(target)
        print(f"\n[INFO] Domain: {target}")
        print(f"[INFO] Resolved IP: {ip_addr}")
        return ip_addr
    except Exception:
        print("[ERROR] Unable to resolve domain.")
        return None


# ─────────────────────────────────────────────
# ARGUMENT OPTIONS
# ─────────────────────────────────────────────
def list_options():
    print("\nAvailable Options:")
    print("  count   → Number of packets")
    print("  size    → Packet size")
    print("  ttl     → Time to live")
    print("  timeout → Timeout duration")
    print("  flood   → Continuous ping")
    print("  df      → Don't fragment flag")


# ─────────────────────────────────────────────
# COMMAND GENERATOR
# ─────────────────────────────────────────────
def create_command(target, option, val):
    system_os = platform.system().lower()

    mapping = {
        "count": ("-n", "-c"),
        "size": ("-l", "-s"),
        "ttl": ("-i", "-i"),
        "timeout": ("-w", "-W"),
        "flood": ("-t", "-c 50"),
        "df": ("-f", "-M do"),
    }

    if option not in mapping:
        print("[ERROR] Invalid option selected.")
        return None

    win_flag, linux_flag = mapping[option]

    if system_os == "windows":
        if option == "flood":
            return ["ping", target, win_flag]
        return ["ping", win_flag, str(val), target]

    else:
        if option == "flood":
            return ["ping"] + linux_flag.split() + [target]
        elif option == "df":
            return ["ping"] + linux_flag.split() + [target, "-c", "4"]
        return ["ping", linux_flag, str(val), target]


# ─────────────────────────────────────────────
# EXECUTION ENGINE
# ─────────────────────────────────────────────
def execute_ping(command):
    print("\n[RUNNING]", " ".join(command))
    print("-" * 60)

    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=25)
        output = result.stdout if result.stdout else result.stderr
        print(output)
        return output
    except subprocess.TimeoutExpired:
        print("[ERROR] Execution timed out.")
        return ""
    except Exception:
        print("[ERROR] Ping execution failed.")
        return ""


# ─────────────────────────────────────────────
# REPORT WRITER
# ─────────────────────────────────────────────
def generate_report(target, ip, option, val, result):
    file_name = f"report_{target.replace('.', '_')}_{datetime.datetime.now().strftime('%H%M%S')}.txt"

    with open(file_name, "w") as file:
        file.write("NETWORK PROBE REPORT\n")
        file.write("=" * 50 + "\n")
        file.write(f"Target      : {target}\n")
        file.write(f"IP Address  : {ip}\n")
        file.write(f"Option Used : {option}\n")
        file.write(f"Value       : {val}\n")
        file.write(f"Time        : {datetime.datetime.now()}\n")
        file.write(f"System      : {platform.system()}\n\n")
        file.write("OUTPUT:\n")
        file.write("-" * 50 + "\n")
        file.write(result)

    print(f"\n[INFO] Report saved as: {file_name}")


# ─────────────────────────────────────────────
# MAIN CONTROLLER
# ─────────────────────────────────────────────
def run():
    display_header()

    target = input("\nEnter target domain: ").strip()
    if not target:
        target = "example.com"

    ip = fetch_ip(target)
    if not ip:
        sys.exit()

    list_options()

    opt = input("\nChoose option: ").strip().lower()
    value = None

    if opt not in ["flood", "df"]:
        value = input("Enter value (default=4): ").strip()
        if not value:
            value = "4"

    cmd = create_command(target, opt, value)
    if not cmd:
        sys.exit()

    output = execute_ping(cmd)

    print("\nSUMMARY")
    print("-" * 50)
    print(f"Target : {target}")
    print(f"IP     : {ip}")
    print(f"Mode   : {opt}")
    print("-" * 50)

    save = input("\nSave report? (y/n): ").lower()
    if save == "y":
        generate_report(target, ip, opt, value, output)

    print("\n[✔] Task Completed.\n")


if __name__ == "__main__":
    run()

        NETWORK PROBE TOOL - USING PING
        (Educational Purpose Only)

Enter target domain: 142.251.32.110

[INFO] Domain: 142.251.32.110
[INFO] Resolved IP: 142.251.32.110

Available Options:
  count   → Number of packets
  size    → Packet size
  ttl     → Time to live
  timeout → Timeout duration
  flood   → Continuous ping
  df      → Don't fragment flag

Choose option: count
Enter value (default=4): 4

[RUNNING] ping -c 4 142.251.32.110
------------------------------------------------------------
[ERROR] Ping execution failed.

SUMMARY
--------------------------------------------------
Target : 142.251.32.110
IP     : 142.251.32.110
Mode   : count
--------------------------------------------------

Save report? (y/n): y

[INFO] Report saved as: report_142_251_32_110_104136.txt

[✔] Task Completed.

